<h1 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 700; margin-bottom: 20px;">
Financial Distress Early Warning and Explainable ML System
</h1>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px; line-height: 1.6;">
<b>Author:</b> Eegunjobi Feranmi<br>
<b>Dataset:</b> Financial ratios and recorded bankruptcy outcomes for Polish companies<br>
<b>Tools:</b> Python, Pandas, Scikit learn, Logistic Regression, Gradient Boosted Trees, SHAP
</p>

<hr style="border: 0.5px solid #d1d1d1;">

<h2 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 600;">
Business Problem
</h2>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px; line-height: 1.6;">
Corporate insolvencies are rising worldwide, and global insolvencies rose 10 percent in 2024, putting them 12 percent above the 2016 to 2019 average. For banks, credit insurers and advisory firms, a single corporate failure can mean unrecoverable loans, unpaid claims or investment losses, yet analysts cannot manually review the financial statements of every borrower or investment target. They need an automated early warning system that screens companies at scale and shows why each one is flagged.
</p>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px; line-height: 1.6;">
This project builds a machine learning model that predicts bankruptcy risk from historical financial ratios, comparing a Logistic Regression baseline with a gradient boosted model and using SHAP to explain which ratios drive each company risk score. Because bankruptcies are rare, the models are judged on recall, precision and PR AUC rather than accuracy, and the decision threshold is tuned to balance missed bankruptcies against false alarms. The predicted probabilities are then converted into a risk ranked list so analysts can investigate the highest risk companies first. The prototype tests the approach on a public historical benchmark, and it does not prove why companies fail or that the model will work for present day Nigerian firms.
</p>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px; line-height: 1.6;">
<b>Business Question:</b> Can we predict whether a company is at risk of bankruptcy from its historical financial ratios and explain the reasons, so that analysts can prioritise which companies to investigate first?
</p>

<hr style="border: 0.5px solid #d1d1d1;">

<h2 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 600;">
Dataset
</h2>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px; line-height: 1.6;">
This project uses the Polish Companies Bankruptcy dataset from the UCI Machine Learning Repository, which provides five separate forecasting datasets. Only the first year file is used here, so all results apply to that file and not to the full collection. It contains 7,027 company observations, of which 271 (3.9 percent) went bankrupt during the specified forecast period and 6,756 (96.1 percent) did not, a ratio of roughly 25 healthy companies for every bankrupt one. The data consists of financial ratios drawn from historical company records, principally covering 2000 to 2013.
</p>

<hr style="border: 0.5px solid #d1d1d1;">

<h2 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 600;">
Step 1 — Import Libraries
</h2>

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.io import arff
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

<hr style="border: 0.5px solid #d1d1d1;">

<h2 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 600; text-align: left;">
Step 2 — Load the Data
</h2>

In [10]:
file_path = 'data/raw/1year.arff'

data, meta = arff.loadarff(file_path)
df = pd.DataFrame(data)

# Decode byte strings to normal strings (ARFF loads as bytes)
for col in df.select_dtypes([object]).columns:
    df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Quick check
df.head()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class
0,0.200550,0.37951,0.39641,2.0472,32.3510,0.38825,0.249760,1.33050,1.1389,0.50494,...,0.121960,0.39718,0.87804,0.001924,8.4160,5.1372,82.658,4.4158,7.4277,0
1,0.209120,0.49988,0.47225,1.9447,14.7860,0.00000,0.258340,0.99601,1.6996,0.49788,...,0.121300,0.42002,0.85300,0.000000,4.1486,3.2732,107.350,3.4000,60.9870,0
2,0.248660,0.69592,0.26713,1.5548,-1.1523,0.00000,0.309060,0.43695,1.3090,0.30408,...,0.241140,0.81774,0.76599,0.694840,4.9909,3.9510,134.270,2.7185,5.2078,0
3,0.081483,0.30734,0.45879,2.4928,51.9520,0.14988,0.092704,1.86610,1.0571,0.57353,...,0.054015,0.14207,0.94598,0.000000,4.5746,3.6147,86.435,4.2228,5.5497,0
4,0.187320,0.61323,0.22960,1.4063,-7.3128,0.18732,0.187320,0.63070,1.1559,0.38677,...,0.134850,0.48431,0.86515,0.124440,6.3985,4.3158,127.210,2.8692,7.8980,0


<hr style="border: 0.5px solid #d1d1d1;">

<h2 style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-weight: 600; text-align: left;">
Step 3 — Data Understanding
</h2>

<p style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif; font-size: 15px;">
Before any analysis, I want to understand the structure of the data. This includes checking data types, missing values, duplicates, and basic statistics. This is the first thing a data scientist does when working with a new dataset.
</p>

In [13]:
df.shape

(7027, 65)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7027 entries, 0 to 7026
Data columns (total 65 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Attr1   7024 non-null   float64
 1   Attr2   7024 non-null   float64
 2   Attr3   7024 non-null   float64
 3   Attr4   6997 non-null   float64
 4   Attr5   7019 non-null   float64
 5   Attr6   7024 non-null   float64
 6   Attr7   7024 non-null   float64
 7   Attr8   7002 non-null   float64
 8   Attr9   7026 non-null   float64
 9   Attr10  7024 non-null   float64
 10  Attr11  6988 non-null   float64
 11  Attr12  6997 non-null   float64
 12  Attr13  7027 non-null   float64
 13  Attr14  7024 non-null   float64
 14  Attr15  7025 non-null   float64
 15  Attr16  7002 non-null   float64
 16  Attr17  7002 non-null   float64
 17  Attr18  7024 non-null   float64
 18  Attr19  7027 non-null   float64
 19  Attr20  7027 non-null   float64
 20  Attr21  5405 non-null   float64
 21  Attr22  7024 non-null   float64
 22  

In [18]:
pd.set_option('display.max_columns', None)

df.describe()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,Attr11,Attr12,Attr13,Attr14,Attr15,Attr16,Attr17,Attr18,Attr19,Attr20,Attr21,Attr22,Attr23,Attr24,Attr25,Attr26,Attr27,Attr28,Attr29,Attr30,Attr31,Attr32,Attr33,Attr34,Attr35,Attr36,Attr37,Attr38,Attr39,Attr40,Attr41,Attr42,Attr43,Attr44,Attr45,Attr46,Attr47,Attr48,Attr49,Attr50,Attr51,Attr52,Attr53,Attr54,Attr55,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64
count,7024.000000,7024.000000,7024.000000,6997.000000,7.019000e+03,7024.000000,7024.000000,7002.000000,7026.000000,7024.000000,6988.000000,6997.000000,7027.000000,7024.000000,7.025000e+03,7002.000000,7002.000000,7024.000000,7027.000000,7.027000e+03,5405.000000,7024.000000,7027.000000,6903.000000,7024.000000,7002.000000,6.716000e+03,6993.000000,7024.000000,7027.000000,7027.000000,6989.000000,6997.000000,7002.000000,7024.000000,7024.000000,4287.000000,7024.000000,7027.000000,6997.000000,6943.000000,7027.000000,7.027000e+03,7.027000e+03,6893.000000,6996.000000,6998.000000,7024.000000,7027.000000,7002.000000,7024.000000,6998.000000,6993.000000,6993.000000,7.027000e+03,7.027000e+03,7026.000000,7.027000e+03,7026.000000,6.892000e+03,7005.000000,7.027000e+03,6997.000000,6993.000000
mean,0.034660,0.560215,0.119969,2.629143,-2.631672e+02,0.059712,0.313876,2.623996,5.552855,1.825832,0.354393,0.799824,2.093345,0.313876,1.802696e+03,0.870690,3.751868,0.313876,0.462392,1.162128e+03,10.367516,0.288163,0.423610,0.540231,1.264430,0.831449,1.321989e+03,2.702905,4.194831,23.568949,0.473717,236.571445,7.144093,3.666283,0.306880,5.900730,173.453694,1.911897,0.269342,0.826132,0.604475,0.263638,4.588504e+03,3.426404e+03,-30.208158,1.853339,86.752262,0.148865,-1.367363,2.039821,0.467683,0.584584,98.720579,99.636002,8.855693e+03,-1.577367e+02,0.193243,1.587409e+02,0.277829,4.328830e+02,15.642228,4.763202e+03,8.126852,208.731950
std,4.565504,5.350084,5.275459,13.257356,3.707460e+04,6.051113,8.353274,18.708327,101.995448,33.836452,8.658758,5.316444,159.699448,8.353274,8.392232e+04,5.595342,19.514235,8.353274,30.646874,9.315822e+04,417.358298,8.337193,29.742101,13.501067,28.976962,7.781771,3.550416e+04,27.434030,0.677757,1826.847970,30.647711,4841.844793,17.077782,16.338097,7.565155,101.995601,6339.491580,33.835785,27.064231,12.594535,13.834767,27.071123,3.627309e+05,2.696211e+05,3088.908964,12.882415,1131.891001,7.793534,107.717867,5.039658,5.278947,8.527220,2785.367407,2799.586619,7.247527e+04,1.322125e+04,4.344046,1.322124e+04,6.339149,2.612802e+04,261.554534,3.107835e+05,19.996419,5140.708804
min,-256.890000,-72.162000,-440.500000,0.000000,-2.722100e+06,-397.890000,-189.560000,-141.410000,0.000005,-440.550000,-189.450000,-23.207000,-607.420000,-189.560000,-5.611900e+06,-42.322000,-0.412920,-189.560000,-622.060000,0.000000e+00,-1325.000000,-216.800000,-634.590000,-189.560000,-459.560000,-21.793000,-1.479000e+04,-490.090000,0.176090,-149.070000,-622.060000,0.000000,0.000000,-280.260000,-169.470000,0.000005,-525.520000,-440.550000,-701.630000,-101.270000,-77.791000,-701.630000,0.000000e+00,0.000000e+00,-256230.000000,-101.260000,0.000000,-218.420000,-9001.000000,0.000000,0.000000,0.000000,-130.470000,-122.030000,-8.004700e+05,-1.108300e+06,-315.370000,-4.194000e-03,-327.970000,4.700000e-05,0.000016,0.000000e+00,0.000015,0.000010
25%,0.021182,0.296678,0.026968,1.063100,-4.449800e+01,0.000000,0.028023,0.445710,1.037225,0.300785,0.038218,0.063057,0.038073,0.028023,3.435400e+02,0.122810,1.467025,0.028023,0.014812,1.708300e+01,1.024400,0.020362,0.011253,0.049077,0.195252,0.110262,2.353975e-01,0.050522,3.720875,0.094681,0.019233,48.291000,3.050500,0.247445,0.023120,1.264700,1.296500,0.419380,0.012786,0.052276,0.034762,0.011540,6.525600e+01,3.387300e+01,0.102010,0.610012,18.293000,-0.016980,-0.009992,0.805425,0.216135,0.132110,0.715110,0.972140,9.712000e+01,2.031450e-02,0.056772,8.647650e-01,0.000000,5.923950e+00,4.829000,4.322250e+01,3.425400,2.538600
50%,0.075802,0.482960,0.181275,1.502000,-5.373900e+00,0.000000,0.090109,1.015

In [20]:
(df.isnull().mean() * 100).loc[lambda x: x > 0].sort_values(ascending=False).round(2).to_frame(name='Percentage of Missing Values')

,Percentage of Missing Values
Attr37,38.99
Attr21,23.08
Attr27,4.43
Attr60,1.92
Attr45,1.91
Attr24,1.76
Attr41,1.20
Attr11,0.56
Attr32,0.54
Attr28,0.48


In [21]:
df.duplicated().sum()

np.int64(82)

In [34]:
pd.DataFrame({'Status': ['Not Bankrupt', 'Bankrupt'], 
              'Number of Companies': [(c == 0).sum(), (c == 1).sum()]})

,Status,Number of Companies
0,Not Bankrupt,6756
1,Bankrupt,271
